## Data Generation for Grocery Supply Chain
### Import Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import polars.selectors as cs
import os
import json
import fastparquet

import create_data_functions, weather_conditions

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

pl.set_random_seed(seed=56)

### Paths

In [2]:
# Define data paths
p = Path("data")

raw_data_path = p / 'raw'

processed_path = p / 'processed'

external_data_path = p / 'external'


In [3]:
# List of JSON filenames (without extension) to be loaded
arch_json = ['products','products_categories', 'suppliers']

# Dictionary to store the loaded JSON content
store_catalog = {}

# Loop through each filename, build the full path, and load the JSON data
for name in arch_json:
    file_path = raw_data_path / f"{name}.json"  # Construct full file path
    if file_path.exists():
        with file_path.open("r", encoding="utf-8") as f:     # Open the JSON file
            store_catalog[name] = json.load(f)               # Load and store the data under its name
    else:
        print(f"Notice: File {file_path} was not found.")

# Catalog Information

In [4]:
# Calculate the total number of products in the store catalog
total_products = len(store_catalog["products"])
# Calculate the total number of suppliers in the store catalog
total_suppliers = len(store_catalog["suppliers"])

# Create unique supplier IDs with suffix 'S'
suppliers_id = create_data_functions.create_IDs(total_suppliers, suffix='S')

# Initialize a random number generator with a fixed seed for reproducibility
rng = np.random.default_rng(seed=43)

# Randomly select 15 unique suppliers to be considered "top suppliers"
suppliers_top = rng.choice(list(set(store_catalog['suppliers'].keys())), 15, replace=False)

# Create a LazyFrame of products with product names as a column
catalog_lazy = (
    pl.DataFrame(store_catalog["products"])  # Convert products dictionary to DataFrame
    .transpose(include_header=True)          # Transpose to align product data correctly
    .lazy()                                  # Convert to LazyFrame for deferred execution
    .rename({"column": "product"})           # Rename column to 'product'
    .unnest("column_0")                      # Expand nested column values
    .with_columns(
        pl.Series(
            "product_id",
            create_data_functions.create_IDs(total_products, suffix='P')  # Generate product IDs
        ))
    .select([                                # Select relevant product attributes
        "product_id",
        "product",
        "category",
        "sub_category",
        "shelf_life_days",
        "maximum_days_on_sale",
        "seasonality",
        "storage_recommendation",
        "unit_of_measurement"
        ])

    # Join product data with supplier data
    .join(
        pl.from_dicts(store_catalog["suppliers"])   # Convert suppliers dictionary to DataFrame
        .transpose(include_header=True)             # Transpose to align supplier data correctly
        .unnest("column_0")                         # Expand nested column values
        .with_columns([
            pl.Series("supplier_id", suppliers_id), # Add supplier IDs
            pl.Series("supplier_rating", np.random.randint(1, 6, size=total_suppliers)).cast(pl.UInt8)  # Random ratings
        ])
        .lazy()                                     # Convert to LazyFrame
        .rename({
            "products": "product",                  # Rename 'products' to 'product'
            "column": "supplier"                    # Rename 'column' to 'supplier'
        })
        .explode("product")                         # Expand product lists per supplier
        .select([                                   # Select relevant supplier attributes
            "supplier_id", "supplier_rating", "supplier", "product", "distance_km", "moq"
        ])
    ,
    on="product",                                   # Join on product field
    how="inner"                                     # Inner join to match products with suppliers
    )
)

## Simulating Lead Time with Urban, Highway, and Off-Road Segments

In this notebook, we simulate delivery lead times by splitting each supplier route into three components:

- **Urban segment**: Always the first 50 km of the journey. This represents city traffic and local distribution.
- **Highway segment**: The remaining distance after the urban portion, typically used for processed or industrial goods.
- **Off-road segment**: A fraction of the remaining distance, applied only to categories that logically originate from rural or agricultural areas (e.g., Meat, Seafood, Vegetables, Fruits, Dairy, Eggs, Grains & Rice, Dried Fruits).

### Simulation Logic
1. **Urban distance** = `min(distance_km, 50)`
2. **Remaining distance** = `max(distance_km - 50, 0)`
3. For off-road categories:
   - Assign a random proportion between 10% and 40% of the remaining distance to off-road.
   - The rest becomes highway distance.
4. For other categories:
   - Off-road distance = 0
   - Highway distance = remaining distance

This approach ensures realistic variability:
- Short routes (≤ 50 km) are fully urban.
- Longer routes for fresh/agricultural products include rural access roads.
- Processed goods rely on highways and urban distribution.

The resulting dataset will contain three new columns:
- `urban_km`
- `highway_km`
- `off_road_km`

These can be used to model delivery times, costs, or reliability depending on the type of route.


In [5]:
# Apply the road simulation function to enrich the catalog 
catalog_lazy = create_data_functions.road_simulation_polars(catalog_lazy)

## Meteorological Data for Supply Chain Management

In [6]:
# Set the path to the external weather data source
# Source: https://bdmep.inmet.gov.br/

# Set the path to the weather CSV file
# archive_csv = external_data_path + 'dados_83967_D_2015-01-01_2025-09-18.csv'
archive_csv = external_data_path / 'dados_B807_D_2022-12-07_2025-09-22.csv'

# Rename columns to clear and descriptive English names
columns_name = {
    "Data Medicao": "measurement_date",
    "PRECIPITACAO TOTAL, DIARIO (AUT)(mm)": "daily_total_precipitation_mm",
    "TEMPERATURA MAXIMA, DIARIA (AUT)(°C)": "daily_maximum_temperature_c",
    "TEMPERATURA MINIMA, DIARIA (AUT)(°C)": "daily_minimum_temperature_c",
    "VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)": "daily_average_wind_speed_mps"
}

In [7]:
# Read the CSV file into a LazyFrame for efficient query planning
weather_lazy = (
    pl.scan_csv(
        archive_csv,
        separator=";",          # Use semicolon as delimiter
        decimal_comma=True,     # Interpret comma as decimal separator
        skip_rows=10,           # Skip metadata/header rows at the top
        try_parse_dates=True    # Attempt automatic date parsing
    )
    # Rename columns to clear and descriptive English names
    .rename(columns_name)

    # Drop empty column (created by trailing delimiter in CSV)
    .drop("")

    # Replace string "null" with actual None values and cast to Float32
    .with_columns([
        pl.when(pl.col(c) == "null")
        .then(None)             # Convert "null" string to None
        .otherwise(pl.col(c))   # Keep original value otherwise
        .cast(pl.Float32)       # Cast to float for numeric operations
        .alias(c)               # Preserve column name
        for c in columns_name.values() if c != "measurement_date"
    ])

    # Remove rows where all values are null (fully missing records)
    .filter(~pl.all_horizontal(pl.all().is_null()))

    # Remove the first two rows (index 0 and 1) to clean dataset
    .slice(2, None)

    .with_columns(
        # Create binary flags indicating missing values for each variable
        pl.col("daily_total_precipitation_mm").is_null().alias("precipitation_missing"),
        pl.col("daily_maximum_temperature_c").is_null().alias("max_temp_missing"),
        pl.col("daily_minimum_temperature_c").is_null().alias("min_temp_missing"),
        pl.col("daily_average_wind_speed_mps").is_null().alias("wind_missing"),

        # Impute missing values using forward fill (propagate last valid observation forward)
        pl.col("daily_total_precipitation_mm").fill_null(strategy="forward"),
        pl.col("daily_maximum_temperature_c").fill_null(strategy="forward"),
        pl.col("daily_minimum_temperature_c").fill_null(strategy="forward"),
        pl.col("daily_average_wind_speed_mps").fill_null(strategy="forward"),
    )

    .with_columns(
        # Impute missing values using backward fill (propagate next valid observation backward)
        pl.col("daily_total_precipitation_mm").fill_null(strategy="backward"),
        pl.col("daily_maximum_temperature_c").fill_null(strategy="backward"),
        pl.col("daily_minimum_temperature_c").fill_null(strategy="backward"),
        pl.col("daily_average_wind_speed_mps").fill_null(strategy="backward"),
    )

    # Select final set of columns: original values + missing flags
    .select(
        [
            'measurement_date',                # Keep the measurement date column
            'daily_total_precipitation_mm',    # Original precipitation values
            'precipitation_missing',           # Binary flag for missing precipitation
            'daily_maximum_temperature_c',     # Original maximum temperature values
            'max_temp_missing',                # Binary flag for missing max temperature
            'daily_minimum_temperature_c',     # Original minimum temperature values
            'min_temp_missing',                # Binary flag for missing min temperature
            'daily_average_wind_speed_mps',    # Original wind speed values
            'wind_missing'                     # Binary flag for missing wind speed
        ]
    ) 
)

In [8]:
# Collect the weather data into a DataFrame
df_weather = weather_lazy.collect()

# Find the minimum measurement date
date_min = df_weather["measurement_date"].min()

# Find the maximum measurement date
date_max = df_weather["measurement_date"].max()

# Generate the expected sequence of dates from min to max, with daily intervals
expect_seq = pl.date_ranges(date_min, date_max, interval="1d", eager=True).explode()

# Get the unique measurement dates from the dataset and sort them
exist_date = df_weather["measurement_date"].unique().sort()

# Check if the existing dates match the expected sequence (detect gaps)
gaps = not exist_date.equals(expect_seq)

# Print whether there are date gaps
print(f"Have date gaps? {gaps}")

Have date gaps? False


In [9]:
# Get schema without collecting the full dataset
print(weather_lazy.collect_schema())

Schema([('measurement_date', Date), ('daily_total_precipitation_mm', Float32), ('precipitation_missing', Boolean), ('daily_maximum_temperature_c', Float32), ('max_temp_missing', Boolean), ('daily_minimum_temperature_c', Float32), ('min_temp_missing', Boolean), ('daily_average_wind_speed_mps', Float32), ('wind_missing', Boolean)])


In [10]:
# Polars optimization
print(weather_lazy.explain())

simple π 9/9 ["measurement_date", ... 8 other columns]
   WITH_COLUMNS:
   [col("daily_total_precipitation_mm").fill_null_with_strategy(), col("daily_maximum_temperature_c").fill_null_with_strategy(), col("daily_minimum_temperature_c").fill_null_with_strategy(), col("daily_average_wind_speed_mps").fill_null_with_strategy()] 
     WITH_COLUMNS:
     [col("daily_total_precipitation_mm").is_null().alias("precipitation_missing"), col("daily_maximum_temperature_c").is_null().alias("max_temp_missing"), col("daily_minimum_temperature_c").is_null().alias("min_temp_missing"), col("daily_average_wind_speed_mps").is_null().alias("wind_missing"), col("daily_total_precipitation_mm").fill_null_with_strategy(), col("daily_maximum_temperature_c").fill_null_with_strategy(), col("daily_minimum_temperature_c").fill_null_with_strategy(), col("daily_average_wind_speed_mps").fill_null_with_strategy()] 
      SLICE[offset: 2, len: 18446744073709551615]
        FILTER [([([([(col("measurement_date").is_not_nu

### Define Weather Severity Levels

In [11]:
# Apply the weather classification function to the cleaned DataFrame to generate severity and category labels
weather_analyser = weather_conditions.PolarsWeatherConditions(weather_lazy)
weather_severity_lazy = (
    weather_analyser.classify_weather()
    .select([
            'measurement_date',
            'temperature_classification',
            'precipitation_classification',
            'wind_classification', 
            'weather_severity'
        ])
    .rename({"measurement_date": "received_date"})
)

In [12]:
# Show 10 samples rows of the DataFrame
weather_severity_lazy.head(3).collect()

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,str,str,str,str
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-10,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-11,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""


In [13]:
# Collects and returns the schema definition 
weather_severity_lazy.collect_schema()

Schema([('received_date', Date),
        ('temperature_classification', String),
        ('precipitation_classification', String),
        ('wind_classification', String),
        ('weather_severity', String)])

# Realistic supply chain modeling based on Weather, Product and Seasonality

In [14]:
# Determine the number of samples based on the length of the weather DataFrame
n_samples = len(weather_severity_lazy.collect())

# Calculate how many times we need to replicate the weather data to reach ~300,000 rows
quantity_rows = 1000
multiply_rows = quantity_rows // n_samples + 1
n_total = multiply_rows * n_samples

# Replicate the fixed weather conditions multiply_rows times
weather_replicated_lazy = pl.concat([weather_severity_lazy] * multiply_rows)

# Sample products with replacement to match the total number of rows
catalog_sampled_lazy = catalog_lazy.collect().sample(n=n_total, with_replacement=True)

# Horizontally merge weather and product data row-by-row, then sort by received_date
merged_prod_weather_lazy = (
    weather_replicated_lazy
    .collect()
    .hstack(catalog_sampled_lazy)
    .sort("received_date")
    .with_columns(
        (pl.col("received_date").dt.strftime("%B").is_in(pl.col("seasonality")).alias("in_season"))
    )
).lazy()

In [15]:
merged_prod_weather_lazy.show(3)

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity,product_id,product,category,sub_category,shelf_life_days,maximum_days_on_sale,seasonality,storage_recommendation,unit_of_measurement,supplier_id,supplier_rating,supplier,distance_km,moq,off_road_km,highway_km,urban_km,in_season
date,str,str,str,str,str,str,str,str,i64,i64,list[str],str,str,str,u8,str,i64,i64,i32,i64,i64,bool
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1543794|P""","""Tomato""","""Fresh Foods""","""Vegetables""",7,3,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""","""1649464|S""",2,"""ValleyFresh Farms""",85,100,13,22,50,false
2022-12-10,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1079566|P""","""Pasta""","""Pantry""","""Grains & Rice""",730,180,[],"""Room Temperature""","""lb""","""1200117|S""",5,"""Pasta Paradise""",130,100,11,69,50,false
2022-12-11,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1823897|P""","""Salmon Fillet""","""Fresh Foods""","""Seafood""",2,1,[],"""Refrigerated""","""lb""","""1501694|S""",5,"""Seafood Select""",195,28,26,119,50,false


In [16]:
print(merged_prod_weather_lazy.collect_schema())

Schema([('received_date', Date), ('temperature_classification', String), ('precipitation_classification', String), ('wind_classification', String), ('weather_severity', String), ('product_id', String), ('product', String), ('category', String), ('sub_category', String), ('shelf_life_days', Int64), ('maximum_days_on_sale', Int64), ('seasonality', List(String)), ('storage_recommendation', String), ('unit_of_measurement', String), ('supplier_id', String), ('supplier_rating', UInt8), ('supplier', String), ('distance_km', Int64), ('moq', Int64), ('off_road_km', Int32), ('highway_km', Int64), ('urban_km', Int64), ('in_season', Boolean)])


## Generate data about holidays, weekdays of the year.

In [ ]:
# Classify each record’s received_date into day types (weekday, weekend, holiday)
# for Brazil, producing a lazy DataFrame with seasonality context.
prod_seasonality_lazy = (
    create_data_functions.day_classification_lazy(
        df=merged_prod_weather_lazy,
        col="received_date",
        country="br"
    )
)


## Generate data for stock quantities and sales volumes.

In [18]:
# Generate sales demand
data_stock_sales_lazy = create_data_functions.classify_grocery_demand_polars(df=prod_seasonality_lazy, columns_date="received_date", country="br")

# Generate sales volume
data_stock_sales_lazy = create_data_functions.simulate_sales_volume_polars(df= data_stock_sales_lazy)

### Simulates speed distributions for different road types and calculate total transit in minute

In [19]:
# Generate normal transit time conform road types
data_transit_time_lazy = create_data_functions.simulate_distribution_speeds(data_stock_sales_lazy)

### Generate stock informations

In [23]:

# Generate transit time and days delivery (decimals)
data_stock_time_lazy = create_data_functions.estimate_delivery_polars(df= data_transit_time_lazy)

# Generate Min_Max Stock
data_stock_sales_lazy = create_data_functions.min_max_stock_polars(df = data_stock_time_lazy)

### Create stock distribution

In [ ]:
# Generate a stock distribution DataFrame from the raw sales data,
# then preview the first 3 rows to quickly inspect the structure.
data_stock_dist_lazy = create_data_functions.create_stock_distribution_polars(data_stock_sales_lazy)

data_stock_dist_lazy.show(3)

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity,product_id,product,category,sub_category,shelf_life_days,maximum_days_on_sale,seasonality,storage_recommendation,unit_of_measurement,supplier_id,supplier_rating,supplier,distance_km,moq,in_season,is_holiday,day_classification,is_weekend,sales_demand,sales_volume,transit_time,delivery_days,min_stock,max_stock,stock_quantity
date,str,str,str,str,str,str,str,str,i64,i64,list[str],str,str,str,u8,str,i64,i64,bool,bool,str,bool,str,i64,f64,f64,i32,i32,i32
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1543794|P""","""Tomato""","""Fresh Foods""","""Vegetables""",7,3,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""","""1649464|S""",2,"""ValleyFresh Farms""",85,100,false,false,"""Weekday""",false,"""High""",154,2.472197,1.437825,223,323,0
2022-12-10,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1079566|P""","""Pasta""","""Pantry""","""Grains & Rice""",730,180,[],"""Room Temperature""","""lb""","""1200117|S""",5,"""Pasta Paradise""",130,100,false,false,"""Saturday""",true,"""High""",62,3.498753,1.193533,70,170,164
2022-12-11,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1823897|P""","""Salmon Fillet""","""Fresh Foods""","""Seafood""",2,1,[],"""Refrigerated""","""lb""","""1501694|S""",5,"""Seafood Select""",195,28,false,false,"""Sunday""",true,"""High""",15,3.6478,0.726544,12,40,17


### Simulate purchase order

In [ ]:
def create_purchase_order_polars(df: pl.DataFrame | pl. LazyFrame, seed: int = None) -> pl.DataFrame | pl. LazyFrame:
    
    # Get the number of rows in the lazy DataFrame
    length = df.collect().height

    # Create a random number generator
    rng = np.random.default_rng(seed=seed)

    # Shelf life (perishable)
    ultra_perishible = rng.uniform(low=0, high=1, size=length) # <= 7 days
    fast_perishible = rng.uniform(low=2, high=4, size=length) # <= 14 days
    stable_perishible = rng.uniform(low=5, high=12, size=length) # <= 60 days
    long_perishible = rng.uniform(low=15, high=45, size=length) # > 60 days

    # Season factor
    season_factor = rng.uniform(low=0.8, high=0.9, size=length)


    return (
        df.with_columns(
            pl.Series("ultra_perishible", ultra_perishible),
            pl.Series("fast_perishible", fast_perishible),
            pl.Series("stable_perishible", stable_perishible),
            pl.Series("long_perishible", long_perishible),
            pl.Series("season_factor", season_factor)
        )
        # Mitigation factor
        # beta = (R - 1) x 0.15 + 0.7
        .with_columns(
            pl.when(pl.col("supplier_rating") < 3).then(0.0)
            .otherwise(
            (pl.col("supplier_rating") - 1)
            * pl.lit(0.15)
            + pl.lit(0.7))
            .alias("beta_factor")
            )
        # aplica resiliência
        # weather_resilience = 1 + (weather_severity x beta_factor)
        .with_columns(
            pl.when(pl.col("weather_severity") == "Normal")
            .then(pl.col("w_normal")
                  * pl.col("beta_factor") + pl.lit(1))
            .when(pl.col("weather_severity") == "Moderate")
            .then(pl.col("w_moderate")
                  * pl.col("beta_factor") + pl.lit(1))
            .when(pl.col("weather_severity") == "Severe")
            .then(pl.col("w_severe")
                  * pl.col("beta_factor") + pl.lit(1))
            .otherwise(
                pl.lit(0.0)
            ).alias("weather_resilience_factor")
        )
        # Create lead time base using shelf life days
        .with_columns(
            (
                pl.when(pl.col("shelf_life_days") <= 7)
                .then(pl.col("ultra_perishible"))
                .when(pl.col("shelf_life_days") <= 14)
                .then(pl.col("fast_perishible"))
                .when(pl.col("shelf_life_days") <= 60)
                .then(pl.col("stable_perishible"))
                .otherwise(pl.col("long_perishible"))
            )
            .round()
            .alias("lt_base")
        )
        .with_columns(
            # Create noise if weekend + 1, if holiday + 1.5 weekend + holiday + 1.8
            (   
                pl.when(pl.col("is_weekend") & pl.col("is_holiday"))
                .then(1.8)
                .when(pl.col("is_holiday"))
                .then(1.5)
                .when(pl.col("is_weekend"))
                .then(1.0)
                .otherwise(0.0)
            ).alias("noise_calendar")
        )
        .with_columns(
            (
                pl.when(pl.col("noise_calendar") < 1.0)
                .then(pl.lit(rng.lognormal(mean=1.0, sigma= 0.1)))
                .when(pl.col("noise_calendar") < 1.5)
                .then(pl.lit(rng.lognormal(mean=1.0, sigma= 0.2)))
                .when(pl.col("noise_calendar") < 1.8)
                .then(pl.lit(rng.lognormal(mean=1.5, sigma= 0.3)))
                .otherwise(pl.lit(rng.lognormal(mean=1.8, sigma= 0.5)))
                .floor().cast(pl.Int8)
            ).alias("noise_calendar_add")
        )
        # calculate lead_time
        .with_columns(
            (
                (pl.col("lt_base") * pl.col("season_factor"))
                * (1 + pl.col("weather_resilience_factor"))
                + pl.col("noise_calendar_add")
                # + supplier_add
                
            ).alias("lead_time")
        )
    ).drop(['received_date', "product", "product_id", "category", "sub_category", "maximum_days_on_sale", "unit_of_measurement", "supplier", "supplier_id", "distance_km", "moq", "storage_recommendation", "temperature_classification", "precipitation_classification", "wind_classification", "day_classification", "sales_demand", "sales_volume", "min_stock", "max_stock", "stock_quantity", "seasonality", "avg_sales"])


In [45]:

create_lead_time_polars(data_stock_dist_lazy).collect()

ColumnNotFoundError: "avg_sales" not found

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'select' <---
 WITH_COLUMNS:
 [[([([(col("lt_base")) * (col("season_factor"))]) * ([(1.0) + (col("weather_resilience_factor"))])]) + (col("noise_calendar_add").cast(Float64))].alias("lead_time")] 
   WITH_COLUMNS:
   [when([(col("noise_calendar")) < (1.0)]).then(dyn float: 2.4362429189096946).otherwise(when([(col("noise_calendar")) < (1.5)]).then(dyn float: 2.272226537400466).otherwise(when([(col("noise_calendar")) < (1.8)]).then(dyn float: 7.116319747929568).otherwise(dyn float: 6.180467856850662))).floor().cast(Int8).alias("noise_calendar_add")] 
     WITH_COLUMNS:
     [when([(col("is_weekend")) & (col("is_holiday"))]).then(dyn float: 1.8).otherwise(when(col("is_holiday")).then(dyn float: 1.5).otherwise(when(col("is_weekend")).then(dyn float: 1).otherwise(dyn float: 0))).alias("noise_calendar")] 
       WITH_COLUMNS:
       [when([(col("shelf_life_days")) <= (7)]).then(col("ultra_pe

In [ ]:
rng.poisson(3, 10)

In [ ]:
# aplicado no lead_time
supplier_rating # gemini quanto menor melhor 1 a 5 $$\beta = 0.7 + (R - 1) \times 0.15$$

[in_season, delivery_days]# melhora
[is_holiday,	is_weekend] # pioram

1. Resiliência Climática (Fator de Mitigação)Fornecedores de ponta possuem frotas melhores, roteamento dinâmico e priorizam clientes-chave.  
Quando ocorre uma condição climática severa, eles conseguem absorver parte do impacto logístico, enquanto o fornecedor ruim agrava o problema por falta de planejamento.  
Você pode criar um coeficiente de mitigação ($\beta$) atrelado ao supplier_rating ($R$), onde $R \in \{1, 2, 3, 4, 5\}$:$$\beta = 0.7 + (R - 1) \times 0.15$$Fornecedor 1 (O Melhor): $\beta = 0.7$ (Consegue reduzir o impacto do atraso climático em 30%).Fornecedor 3 (Médio): $\beta = 1.0$ (Sofre o atraso calculado integralmente).Fornecedor 5 (O Pior): $\beta = 1.3$ (Não tem contingência; o clima agrava o atraso em mais 30%).Na sua fórmula do lead_time, a aplicação fica assim:$$LT_{real} = LT_{base} \times (1 + (\epsilon \times \beta))$$2. Consistência Operacional (Ruído em Dias Normais)Em dias de clima Normal, o $\epsilon$ climático é quase zero. No entanto, um fornecedor nota 4 ou 5 frequentemente atrasa por desorganização interna (falta de produto, caminhão quebrado, erro de emissão de nota). O fornecedor nota 1 entrega exatamente no $LT_{base}$.Você pode injetar um ruído logístico ($\gamma$) no tempo de entrega, que só pune os piores fornecedores. Se você estiver gerando isso no Polars ou Julia, pode usar uma distribuição onde o desvio padrão cresce conforme a nota piora:Para $R = 1$ e $2$: Ruído zero. A entrega é cirúrgica. $\gamma = 0$Para $R \ge 3$: Sorteie dias de atraso extras usando uma distribuição de Poisson ou Uniforme parametrizada pelo rating. Exemplo:$$\gamma \sim \text{Poisson}(\lambda = R - 2)$$A fórmula final e definitiva do tempo de entrega seria:$$LT_{real} = \left[ LT_{base} \times (1 + (\epsilon \times \beta)) \right] + \gamma$$

A Equação Mestra do Lead TimeAgora temos a arquitetura completa do seu gerador sintético, unindo todas as regras de negócios que construímos até aqui. 
A esteira de cálculo para cada linha do seu dataset fica assim:$$LT_{real} = \underbrace{\left[ \overbrace{(LT_{base} \times S)}^{\text{1. Safra}} \times \overbrace{(1 + (\epsilon \times \beta))}^{\text{2. Clima e Resiliência}} \right]}_{\text{Tempo em Trânsito}} + \underbrace{\gamma_{rating}}_{\text{3. Atraso Fornecedor}} + \underbrace{\omega_{calendario}}_{\text{4. Fila de Doca}}$$Onde:$S$: Multiplicador de in_season (ex: $0.85$ ou $1.0$).$\epsilon$: Multiplicador Log-Normal da condição climática.$\beta$: Mitigador de clima baseado no rating do fornecedor.$\gamma_{rating}$: Atraso logístico interno via Poisson baseada na nota do fornecedor.$\omega_{calendario}$: Dias extras aguardando doca/trânsito via Poisson baseada no calendário.O Impacto Oculto nos Dados (O Pulo do Gato)Para que seu dataset sintético seja "nível Kaggle" em termos de realismo, adicione uma interação entre o tempo de prateleira e a doca.Produtos altamente perecíveis (como pescados ou vegetais folhosos com validade de 2 a 5 dias) que caírem na malha da Poisson de fim de semana/feriado ($\omega \ge 2$) têm uma probabilidade massiva de perderem a validade aguardando a doca abrir.Ao gerar seus dados, se $\omega$ for alto para itens de shelf life curto, dispare sua flag de write-off (descarte total), registre a perda de OTIF e lance o impacto financeiro. Isso forçará qualquer modelo treinado com esses dados a aprender que nunca se deve agendar entregas de ultra-perecíveis em vésperas de feriados.

Alfabeto grego:

alfa (Α, α), beta (Β, β), gama (Γ, γ), delta (Δ, δ), épsilon (Ε, ε), zeta (Ζ, ζ), eta (Η, η), teta (Θ, θ), iota (Ι, ι), kappa (Κ, κ), lambda (Λ, λ), mi (Μ, μ), ni (Ν, ν), xi (Ξ, ξ), ômicron (Ο, ο), pi (Π, π), rho (Ρ, ρ), sigma (Σ, σ/ς), tau (Τ, τ), upsilon (Υ, υ), phi (Φ, φ), chi (Χ, χ), psi (Ψ, ψ) e ômega (Ω, ω)

In [ ]:
s= data_stock_dist_lazy.collect().drop(['received_date', "product", "product_id", "category", "sub_category", "maximum_days_on_sale", "unit_of_measurement", "supplier", "supplier_id", "distance_km", "moq", "storage_recommendation", "temperature_classification", "precipitation_classification", "wind_classification", "day_classification", "sales_demand", "sales_volume", "min_stock", "max_stock", "stock_quantity", "seasonality", "avg_sales"])
s.head(1)

In [ ]:
s.select(pl.col("supplier_rating")).sort("supplier_rating").unique()

### Standardized columns

In [ ]:
standardized_columns = ['received_date', 'lpo', 'lead_time', 'in_season', 'product',
       'product_id', 'category', 'sub_category', 'shelf_life_days',
       'maximum_days_on_sale', 'unit_of_measurement', 'supplier_rating',
       'supplier', 'supplier_id', 'distance_km', 'moq',
       'storage_recommendation', 'temperature_classification',
       'precipitation_classification', 'wind_classification',
       'weather_severity', 'day_classification', 'is_holiday', 'is_weekend',
       'sales_demand', 'sales_volume', 'delivery_days', 'min_stock',
       'max_stock', 'stock_quantity']

final_df = data_purchase_lazy.select(pl.col(standardized_columns))

In [ ]:
s = final_df.collect().drop(['received_date', "lpo", "product", "product_id", "category", "sub_category", "maximum_days_on_sale", "unit_of_measurement", "supplier", "supplier_id", "distance_km", "moq", "storage_recommendation", "temperature_classification", "precipitation_classification", "wind_classification", "day_classification", "sales_demand", "sales_volume", "min_stock", "max_stock", "stock_quantity"])